In [119]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from collections import defaultdict
import math
import collections
import itertools
import numpy as np
import os

# Rendering details
import plotly.io as pio
pio.renderers.default = "browser"

# Alternative formats - comment out as needed
import pyarrow as pa
import pyarrow.feather as feather
import pyarrow.parquet as pq

# Visualizing Cell-Cell Communication in DominoSignal Output

This notebook visualizes the output of a DominoSignal run on a chosen dataset.

The types of plots generated are split into two sections - first, there are dotplots that show the interaction between ligand/receptor pairs, sending/receiving cluster pairs, and the level of ligand and receptor expression. Next, there are a series of River plots/Sankey diagrams that visualize the cell-cell communication pathway for chosen ligands, receptors and transcription factors.

The notebook is split into sections that match those kinds of plots, with an intial section containing helper functions that enable the plotting.


### Helper Functions

These functions performing loading and type conversions used in the rest of the notebook.

In [120]:
# Helper functions

### LOAD ###
def load_domino_df(domino_file):
    filename, file_extension = os.path.splitext(domino_file)
    if file_extension == '.csv':
        domino_df = pd.read_csv(domino_file)
    elif file_extension == '.feather':
        domino_df = pd.read_feather(domino_file)
    elif file_extension == '.parquet':
        domino_df = pd.read_parquet(domino_file, engine='pyarrow')
    else:
        print("Incorrect file type.")
        domino_df = pd.DataFrame([])
    return domino_df

### CONVERT ###
def get_domino_list(domino_df):
    return domino_df.values.tolist()
    

#### River Plot Helper Functions

This is the main body of the function that generates river plots/sankey diagrams, depending on the intended use case.

In [121]:
### River plotting ###
def domino_river_plot_helper(mode, node_label, source_node, target_node, weight, line_label, selection, nodelinecolor="black", nodelinewidth=0.1, nodecolor="midnightblue", linkcolor="rgba(0, 0, 20, 0.01)", linkhovercolor="black", show_selection = False, show_title = True):
    # mode = tf, ligand, receptor, selected
    # set the plot layout + titles based on the mode
    if mode != 'selected':
        if mode == 'tf':
            columns=["Sending Cluster","Ligand","Receptor", "Receiving Cluster"]
            title = 'Selected TF: ' + selection
        elif mode == 'ligand':
            columns=["Sending Cluster","Receptor","TF", "Receiving Cluster"]
            title = "Selected ligand: "+ selection
        elif mode == 'receptor':
            columns=["Sending Cluster","Ligand","TF", "Receiving Cluster"]
            title = "Selected receptor: "+ selection
        else:
            print("Invalid plotting mode")
            return

        if show_title == False:
            title = "Cell-cell communication"

    # multiple selections
    else:
        selected_ligands = selection[0]
        selected_receptors = selection[1]
        selected_tfs = selection[2]
        selected_sending = selection[3]
        selection_receiving = selection[4]

        columns = ["Sending Cluster","Ligand", "Receptor","TF", "Receiving Cluster"]
        if show_selection:
            title = "Selection = Sending Cluster: " + str(selected_sending) + ", Receiving Clusters" + str(selected_receiving) + ", TFs: " + str(selected_tfs) + ", Ligands: " + str(selected_ligands)  + ", Receptors: " + str(selected_receptors)
        else:
            title = "Cell-cell communication"
    
    # Plot based on inputs and set columns/titles
    river_fig_tracked = go.Figure(data=[go.Sankey(
        node = dict(
            line = dict(color = nodelinecolor, width = nodelinewidth),
            label = node_label,
            color = nodecolor
        ),
        link = dict(
            source = source_node, # indices correspond to labels
            target = target_node,
            value = weight,
            label= line_label,
            color = linkcolor,
            hovercolor= linkhovercolor,
      ))])
    
    for x_coordinate, column_name in enumerate(columns):
      river_fig_tracked.add_annotation(
              x=x_coordinate,
              y=1.05,
              xref="x",
              yref="paper",
              text=column_name,
              showarrow=False,
              font=dict(
                  # family="Courier New, monospace",
                  size=16,
                  color="black"
                  ),
              align="center",
              )
    
    river_fig_tracked.update_layout(
        title_text=title,
        font_size=10,
        xaxis={
          'showgrid': False, # thin lines in the background
          'zeroline': False, # thick line at x=0
          'visible': False,  # numbers below
        },
        yaxis={
          'showgrid': False, # thin lines in the background
          'zeroline': False, # thick line at x=0
          'visible': False,  # numbers below
        },
        # Set background
        plot_bgcolor='rgba(0,0,0,0)'
    )
    river_fig_tracked.show()   

def determine_opacity(line_label):

    density = len(line_label)
    if density == 0:
        density = 1
    opacity = min(1.0, 100.0/density)

    if opacity < 0.01:
        opacity = 0.01
    
    return opacity

## Loading DominoSignal Objects

The DominoSignal output object for the chosen dataset is loaded here, in any of the following three file formats: .csv, .feather, and .parquet. By default, .parquet files are used for their reduced side.

In [122]:
# Read in domino dataframe
# domino_file = './domino_network.reformatted.parquet'
domino_file = './domino_network_p21.reformatted.parquet'
domino_df = load_domino_df(domino_file)

## Plotting Parameters

The selection of which TFs, ligands, receptors - or any combination of these - that are plotted are set here. There are plots that show the entirety of the cell-cell communication output when fixed on only one of these values, and there are plots that take a selection of several values and only display the set of interactions that overlap with this selection.

In [123]:
# Set the selected TFs, Ligands and Receptors for these plots - MODIFY AS NEEDED
selected_tf = 'Foxp1'
selected_ligand = 'Gdf11'
selected_receptor = "Antxr1"

# Selecting multiple parameters at once
selected_tfs = ['Atf3']
selected_ligands = ['Tgfb1', 'Gdf11', 'Ubash3b']
selected_receptors = ['Egfr', 'Antxr1']
selected_sending = []
selected_receiving = []

# Helper definitions
Immune_Cells =["B cells", "NK Cells", "T cells", "Dendritic Cells", "Mast Cells", "Macrophages_Pos", "Macrophages_Neg"]
Stromal_Cells = ["Endothelial Cells", "Mesothelial Cells", "Pericytes", "Preadipocytes"]

## Dot Plots

Dot plots allow for the visualization of ligand/receptor pairs, sending/receiving clusters, and the level of expression in each case. The first of these plots is generated for a selected transcription factor, the second is generated for a selected pair of sending and receiving clusters. In order to manage the size of the resulting plots, we also employ a heuristic to reduce the number of plotted ligand/receptor pairs - this is detailed in the third version below.

### Dot Plot - Selected by TF

In [124]:
# Dot plot showing receptor/ligand and sending/receiving pairs, for a selected TF
def domino_dot_plot_tf(domino_df, selected_tf):
    
    domino_output_list = get_domino_list(domino_df)
    
    cluster_pairs = []
    lig_rec_pairs = []
    tfs = []
    
    mean_exp = []
    
    # iterate through and populate
    for line in domino_output_list:
        # Creating the cluster and ligand_receptor pairs
        cluster_key = line[6]+'->'+line[7]
        l_r_key = line[0]+'->'+line[1]
        # Geometric mean computation
        l_r_score = math.sqrt(float(line[3])*float(line[4]))
        cluster_pairs.append(cluster_key)
        lig_rec_pairs.append(l_r_key)
        mean_exp.append(l_r_score)
        tfs.append(line[2])
    
    # Dataframe containing all this data
    dot_chart_data = pd.DataFrame({
        "clusters" : cluster_pairs,
        "lig_rec" : lig_rec_pairs,
        "mean_exp" : mean_exp,
        "tf": tfs
    
    })

    # Select the rows containing the selected_tf
    dot_chart_data_subset = dot_chart_data.loc[dot_chart_data['tf'] == selected_tf]

    # Feed this into plotting

    dot_fig = px.scatter(
        # dot_chart_data,
        dot_chart_data_subset,
        x="clusters",
        y="lig_rec",
        color="mean_exp",              # color based on relationship score
        # size="score",               # optional: size represents score too
        color_continuous_scale="Inferno_r",  # heatmap-like gradient
        title= selected_tf
    )
    
    n_y = dot_chart_data_subset["lig_rec"].nunique()
    n_x = dot_chart_data_subset["clusters"].nunique()
    
    
    # Make it look like a matrix
    dot_fig.update_layout(
        xaxis_title="Sending -> Receiving Cluster",
        yaxis_title="Ligand -> Receptor Pair",
        # Sorting and adjusting the unnecessary space above and blow
        xaxis=dict(categoryorder="category ascending", range=[-0.5, n_x - 0.5]),
        yaxis=dict(categoryorder="category descending", range=[-0.5, n_y - 0.5]),
        # Ticks
        xaxis_tickangle=-90,
        # Specifying dimensions
        height= 30 * dot_chart_data_subset["lig_rec"].nunique(),
        width= 10 * dot_chart_data_subset["clusters"].nunique(),
        #margin=dict(l=20, r=20, t=20, b=20),
        # Legend
        coloraxis_colorbar=dict(
            len=0.5,
            y=0.5
        )
    )
    
    # dot_fig.update_traces(marker=dict(size=14))
    
    dot_fig.show()

In [125]:
domino_dot_plot_tf(domino_df, selected_tf)

### Dot Plot - Selected Sending/Receiving Clusters

In [126]:
def domino_dot_plot(domino_df, sending_clusters, receiving_clusters, save_figure=False, save_location="", filter=False, exp_range_threshold=0.5, exp_max_threshold=1.0):

    # Select only the rows from the domino dataframe that involve either the sending or receiving clusters (if any are specified)
    # Both sending and receiving clusters are specified
    if len(sending_clusters) > 0 and len(receiving_clusters) > 0:
        domino_df_selected = domino_df[(domino_df["sending_cl"].isin(sending_clusters)) & (domino_df["receiving_cl"].isin(receiving_clusters))]
    # Only the sending cluster is specified
    elif len(sending_clusters) > 0 and len(receiving_clusters) == 0:
        domino_df_selected = domino_df[domino_df["sending_cl"].isin(sending_clusters)]
    # Only the receiving cluster is specified
    elif len(sending_clusters) == 0 and len(receiving_clusters) > 0:
        domino_df_selected = domino_df[domino_df["receiving_cl"].isin(receiving_clusters)]
    else:
        return

    # Convert to a list (just for easier processing)
    domino_output_list = get_domino_list(domino_df_selected)

    # All plottable elements
    cluster_pairs = []
    lig_rec_pairs = []
    mean_exp = []
    
    # iterate through list and populate each point
    for line in domino_output_list:
        # Creating the cluster and ligand_receptor pairs
        cluster_key = line[6]+'->'+line[7]
        l_r_key = line[0]+'->'+line[1]
        # Geometric mean computation
        l_r_score = math.sqrt(float(line[3])*float(line[4]))
        # Add this point to the eventual plot
        cluster_pairs.append(cluster_key)
        lig_rec_pairs.append(l_r_key)
        mean_exp.append(l_r_score)

    # Combine this all into a dataframe containing all this data
    dot_chart_data = pd.DataFrame({
        "clusters" : cluster_pairs,
        "lig_rec" : lig_rec_pairs,
        "mean_exp" : mean_exp
    })

    # If thresholding is enabled
    if filter:
        # Goal: get subset of rows that meet chosen thresholds, and only plot those
        # Get sorted list of all unique lig_rec pairs, and iterate through, checking the mean, range and max geom mean for each
        lig_rec_list =  sorted(list(set(dot_chart_data['lig_rec'].tolist())))
        # For each lig/rec pair
        for l_r in lig_rec_list:
            # Get all rows for that ligand-receptor pair
            lig_rec_df = dot_chart_data.loc[dot_chart_data['lig_rec'] == l_r]
            # Get the geometric mean of expression for each row
            lig_rec_gm = lig_rec_df['mean_exp'].tolist()
            # Get the mean across all geom means for this lig_rec
            lig_rec_gm_mean = np.mean(lig_rec_gm)
            # Get the range across this lig_rec
            lig_rec_gm_range = max(lig_rec_gm) - min(lig_rec_gm)
            # Check if any values in this are above the threshold
            lig_rec_gm_max = max(lig_rec_gm)

            # If either of the thresholds is met
            if lig_rec_gm_range > exp_range_threshold or lig_rec_gm_max > exp_max_threshold:
                continue

            else:
                # Remove this lig_rec from dot_chart_data
                dot_chart_data = dot_chart_data[dot_chart_data["lig_rec"] != l_r]

    dot_chart_data_subset = dot_chart_data

    # Feed this into plotting

    dot_fig = px.scatter(
        # dot_chart_data,
        dot_chart_data_subset,
        x="clusters",
        y="lig_rec",
        color="mean_exp",              # color based on relationship score
        # size="score",               # optional: size represents score too
        color_continuous_scale="Inferno_r",  # heatmap-like gradient
    )
    
    n_y = dot_chart_data_subset["lig_rec"].nunique()
    n_x = dot_chart_data_subset["clusters"].nunique()

    if not filter:
        # Make it look like a matrix
        dot_fig.update_layout(
            title="Geometric Mean of Ligand and Receptor Expression",
            xaxis_title="Sending -> Receiving Cluster",
            yaxis_title="Ligand -> Receptor Pair",
            # Sorting and adjusting the unnecessary space above and blow
            xaxis=dict(categoryorder="category ascending", range=[-0.5, n_x - 0.5]),
            yaxis=dict(categoryorder="category descending", range=[-0.5, n_y - 0.5]),
            # Ticks
            xaxis_tickangle=-60,
            # Specifying dimensions
            height= 20 * dot_chart_data_subset["lig_rec"].nunique(),
            # width= 60 * dot_chart_data_subset["clusters"].nunique(),
            #margin=dict(l=20, r=20, t=20, b=20),
            # Legend
            coloraxis_colorbar=dict(
                len=0.5,
                y=0.5
            )
        )
        
    else:
        # Make it look like a matrix
        dot_fig.update_layout(
            title="Geometric Mean of Ligand and Receptor Expression; Expression Range Threshold = " + str(exp_range_threshold) + ", High Expression Threshold = " + str(exp_max_threshold) ,
            xaxis_title="Sending -> Receiving Cluster",
            yaxis_title="Ligand -> Receptor Pair",
            # Sorting and adjusting the unnecessary space above and blow
            xaxis=dict(categoryorder="category ascending", range=[-0.5, n_x - 0.5]),
            yaxis=dict(categoryorder="category descending", range=[-0.5, n_y - 0.5]),
            # Ticks
            xaxis_tickangle=-60,
            # Specifying dimensions
            height= 30 * dot_chart_data_subset["lig_rec"].nunique(),
            # width= 60 * dot_chart_data_subset["clusters"].nunique(),
            #margin=dict(l=20, r=20, t=20, b=20),
            # Legend
            coloraxis_colorbar=dict(
                len=0.5,
                y=0.5
            )
        )
        
    
    dot_fig.update_traces(marker=dict(size=10))

    if save_figure:
        dot_fig.write_image(save_location)
    
    dot_fig.show()

In [127]:
sending_clusters = Stromal_Cells
receiving_clusters = ["Macrophages_Neg", "Macrophages_Pos"]
domino_dot_plot(domino_df, sending_clusters, receiving_clusters, save_figure = False)

### Dot Plots - Additional Thresholding

As seen in the plot above, dot plots for certain sending/receiving sets can be extremely long. To counter this, we have implemented a simple heuristic to prioritise the most "important" rows, generating more manageable plots.

This heuristic considers the geometric mean expression values across each ligand/receptor pair for the sending/receiving clusters it spans, and only includes a row if the range of geometric mean expression values exceeds a selected threshold (`exp_range_threshold`), or if a single value in the row exceeds a selected threshold (`exp_max_threshold`).

In [128]:
domino_dot_plot(domino_df, sending_clusters, receiving_clusters, save_figure = False, filter=True, exp_range_threshold=0.5, exp_max_threshold=1.5)

## River plots/Sankey Diagrams

These plots show the entire flow of signalling from sending to receiving clusters, via the ligands, receptors and TFs involved. To simplify these plots, we first generate versions that fix one of these parameters - a TF, ligand or receptor is fixed, and the communication involving that selected parameter is plotted.

### River Plot for a selected TF

In [129]:
# River plot showing links from sending cluster to ligands to receptors to receiving nodes, for a selected TF
def domino_river_plot_tf(domino_df, selected_tf):

    domino_output_list = get_domino_list(domino_df)

    # Generate list of all nodes in each set
    sending_nodes = sorted(list(set([l[6] for l in domino_output_list])))
    ligand_nodes = sorted(list(set([l[0] for l in domino_output_list])))
    receptor_nodes = sorted(list(set([l[1] for l in domino_output_list])))
    receiving_nodes = sorted(list(set([l[7] for l in domino_output_list])))
    
    # Dictionaries of positions
    sending_nodes_dict = dict(zip(sending_nodes, range(len(sending_nodes))))
    ligand_nodes_dict = dict(zip(ligand_nodes, range(len(ligand_nodes))))
    receptor_nodes_dict = dict(zip(receptor_nodes, range(len(receptor_nodes))))
    receiving_nodes_dict = dict(zip(receiving_nodes, range(len(receiving_nodes))))
    
    # Total list of nodes
    nodes = sending_nodes + ligand_nodes + receptor_nodes + receiving_nodes
    
    # Computed offsets to position
    sending_offset = 0
    ligand_offset = sending_offset + len(sending_nodes)
    receptor_offset = ligand_offset + len(ligand_nodes)
    receiving_offset = receptor_offset + len(receptor_nodes)
    
    # Build graph
    node_label = nodes
    source_node = []
    target_node = []
    weight = []
    # Track which line in the domino object each link comes from
    line_label = []
    # Track each line in the domino object, so we can do the hovering later
    l_c = 0
    
    for line in domino_output_list:
        sending_cl = line[6]
        ligand = line[0]
        receptor = line[1]
        receiving = line[7]
        tf = line[2]
        
        # Populate only for this tf
        if tf != selected_tf:
            continue
    
        # Sending to ligand
        source_node.append(sending_nodes_dict[sending_cl] + sending_offset)
        target_node.append(ligand_nodes_dict[ligand] + ligand_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # Ligand to receptor
        source_node.append(ligand_nodes_dict[ligand] + ligand_offset)
        target_node.append(receptor_nodes_dict[receptor] + receptor_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # Receptor to receiving
        source_node.append(receptor_nodes_dict[receptor] + receptor_offset)
        target_node.append(receiving_nodes_dict[receiving] + receiving_offset)
        weight.append(1)
        line_label.append(l_c)
    
        l_c += 1

    # Feed into plotting

    # Automatically calculate opacity
    calculated_opacity = determine_opacity(line_label)
    calculated_linkcolor = "rgba(255,127,0," + str(calculated_opacity) + ")"

    domino_river_plot_helper('tf', node_label, source_node, target_node, weight, line_label, selection=selected_tf,
                            nodelinecolor="black", nodelinewidth=0.1, nodecolor="blue", linkcolor=calculated_linkcolor, linkhovercolor="blue")

In [130]:
domino_river_plot_tf(domino_df, selected_tf)

### River Plot for a selected ligand

In [131]:
# River plot showing links from sending cluster to receptors to receiving nodes to TFs, for a selected ligand
def domino_river_plot_ligand(domino_df, selected_ligand):
    
    domino_output_list = get_domino_list(domino_df)

    sending_nodes = sorted(list(set([l[6] for l in domino_output_list])))
    receptor_nodes = sorted(list(set([l[1] for l in domino_output_list])))
    receiving_nodes = sorted(list(set([l[7] for l in domino_output_list])))
    tf_nodes = sorted(list(set([l[2] for l in domino_output_list])))
    
    # Dictionaries of positions
    sending_nodes_dict = dict(zip(sending_nodes, range(len(sending_nodes))))
    receptor_nodes_dict = dict(zip(receptor_nodes, range(len(receptor_nodes))))
    receiving_nodes_dict = dict(zip(receiving_nodes, range(len(receiving_nodes))))
    tf_nodes_dict = dict(zip(tf_nodes, range(len(tf_nodes))))
    
    # Total list of nodes
    nodes = sending_nodes + receptor_nodes + tf_nodes + receiving_nodes
    
    # Computed offsets to position
    sending_offset = 0
    receptor_offset = sending_offset + len(sending_nodes)
    tf_offset = receptor_offset + len(receptor_nodes)
    receiving_offset = tf_offset + len(tf_nodes)
    
    # Build graph
    node_label = nodes
    source_node = []
    target_node = []
    weight = []
    # Track which line in the domino object each link comes from
    line_label = []
    # Track each line in the domino object, so we can do the hovering later
    l_c = 0
    
    for line in domino_output_list:
        sending_cl = line[6]
        ligand = line[0]
        receptor = line[1]
        receiving = line[7]
        tf = line[2]
        
        # Populate only for this ligand
        if ligand != selected_ligand:
            continue
    
        # Sending to receptor
        source_node.append(sending_nodes_dict[sending_cl] + sending_offset)
        target_node.append(receptor_nodes_dict[receptor] + receptor_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # receptor to tf
        source_node.append(receptor_nodes_dict[receptor] + receptor_offset)
        target_node.append(tf_nodes_dict[tf] + tf_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # tf to receiving
        source_node.append(tf_nodes_dict[tf] + tf_offset)
        target_node.append(receiving_nodes_dict[receiving] + receiving_offset)
        weight.append(1)
        line_label.append(l_c)
    
        l_c += 1


    # Feed into plotting 
    calculated_opacity = determine_opacity(line_label)
    calculated_linkcolor = "rgba(255,127,0," + str(calculated_opacity) + ")"
    
    domino_river_plot_helper('ligand', node_label, source_node, target_node, weight, line_label, selection=selected_ligand,
                            nodelinecolor="black", nodelinewidth=0.1, nodecolor="blue", linkcolor=calculated_linkcolor, linkhovercolor="blue")
    

In [132]:
domino_river_plot_ligand(domino_df, selected_ligand)

### River Plot for a selected receptor

In [133]:
# River plot showing links from sending cluster to ligands to receiving nodes to TFs, for a selected receptor
def domino_river_plot_receptor(domino_df, selected_receptor):

    domino_output_list = get_domino_list(domino_df)

    sending_nodes = sorted(list(set([l[6] for l in domino_output_list])))
    ligand_nodes = sorted(list(set([l[0] for l in domino_output_list])))
    receiving_nodes = sorted(list(set([l[7] for l in domino_output_list])))
    tf_nodes = sorted(list(set([l[2] for l in domino_output_list])))
    
    # Dictionaries of positions
    sending_nodes_dict = dict(zip(sending_nodes, range(len(sending_nodes))))
    ligand_nodes_dict = dict(zip(ligand_nodes, range(len(ligand_nodes))))
    receiving_nodes_dict = dict(zip(receiving_nodes, range(len(receiving_nodes))))
    tf_nodes_dict = dict(zip(tf_nodes, range(len(tf_nodes))))
    
    # Total list of nodes
    nodes = sending_nodes + ligand_nodes + tf_nodes + receiving_nodes 
    
    # Computed offsets to position
    sending_offset = 0
    ligand_offset = sending_offset + len(sending_nodes)
    tf_offset = ligand_offset + len(ligand_nodes)
    receiving_offset = tf_offset + len(tf_nodes)
    
    # Build graph
    node_label = nodes
    source_node = []
    target_node = []
    weight = []
    # Track which line in the domino object each link comes from
    line_label = []
    # Track each line in the domino object, so we can do the hovering later
    l_c = 0
    
    for line in domino_output_list:
        sending_cl = line[6]
        ligand = line[0]
        receptor = line[1]
        receiving = line[7]
        tf = line[2]
        
        # Populate only for this ligand
        if receptor != selected_receptor:
            continue
    
        # Sending to ligand
        source_node.append(sending_nodes_dict[sending_cl] + sending_offset)
        target_node.append(ligand_nodes_dict[ligand] + ligand_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # ligand to tf
        source_node.append(ligand_nodes_dict[ligand] + ligand_offset)
        target_node.append(tf_nodes_dict[tf] + tf_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # tf to receiving
        source_node.append(tf_nodes_dict[tf] + tf_offset)
        target_node.append(receiving_nodes_dict[receiving] + receiving_offset)
        weight.append(1)
        line_label.append(l_c)
    
        l_c += 1

    # Feed into plotting
    calculated_opacity = determine_opacity(line_label)
    calculated_linkcolor = "rgba(255,127,0," + str(calculated_opacity) + ")"
    
    domino_river_plot_helper('receptor', node_label, source_node, target_node, weight, line_label, selection=selected_receptor,
                            nodelinecolor="black", nodelinewidth=0.1, nodecolor="blue", linkcolor=calculated_linkcolor, linkhovercolor="blue")


In [134]:
domino_river_plot_receptor(domino_df, selected_receptor)

## River Plots - Multiple Selections

In addition to selecting for a single parameter, it may be of interest to select only cell-cell communication pathways that meet several criteria. In this plot, any or all of the sending cluster, ligand, receptor, transcription factor or receiving cluster may be set, and only pathways that involve all selected parameters will be plotted.

In [135]:
def domino_river_plot_selected(domino_df, selected_ligands=[], selected_receptors=[], selected_tfs=[], selected_sending=[], selected_receiving=[]):

    domino_output_list = get_domino_list(domino_df)
    
    # Generate list of all nodes in each set
    sending_nodes = sorted(list(set([l[6] for l in domino_output_list])))
    ligand_nodes = sorted(list(set([l[0] for l in domino_output_list])))
    receptor_nodes = sorted(list(set([l[1] for l in domino_output_list])))
    receiving_nodes = sorted(list(set([l[7] for l in domino_output_list])))
    tf_nodes = sorted(list(set([l[2] for l in domino_output_list])))
    
    # Dictionaries of positions
    sending_nodes_dict = dict(zip(sending_nodes, range(len(sending_nodes))))
    ligand_nodes_dict = dict(zip(ligand_nodes, range(len(ligand_nodes))))
    receptor_nodes_dict = dict(zip(receptor_nodes, range(len(receptor_nodes))))
    receiving_nodes_dict = dict(zip(receiving_nodes, range(len(receiving_nodes))))
    tf_nodes_dict = dict(zip(tf_nodes, range(len(tf_nodes))))
    
    # Total list of nodes
    nodes = sending_nodes + ligand_nodes + receptor_nodes + tf_nodes+ receiving_nodes
    
    # Computed offsets to position
    sending_offset = 0
    ligand_offset = sending_offset + len(sending_nodes)
    receptor_offset = ligand_offset + len(ligand_nodes)
    tf_offset = receptor_offset + len(receptor_nodes)
    receiving_offset = tf_offset + len(tf_nodes)
    
    # Build graph
    node_label = nodes
    source_node = []
    target_node = []
    weight = []
    # Track which line in the domino object each link comes from
    line_label = []
    # Track each line in the domino object, so we can do the hovering later
    l_c = 0
    
    for line in domino_output_list:
        sending_cl = line[6]
        ligand = line[0]
        receptor = line[1]
        receiving_cl = line[7]
        tf = line[2]
        
        # Populate only for the selected subset of tfs, ligands, receptors, sending and receiving clusters
        # if (tf not in selected_tfs and len(selected_tfs) > 0) or (ligand not in selected_ligands and len(selected_ligands) > 0) or (receptor not in selected_receptors and len(selected_receptors) > 0):
        if (tf not in selected_tfs and len(selected_tfs) > 0) or (ligand not in selected_ligands and len(selected_ligands) > 0) or (receptor not in selected_receptors and len(selected_receptors) > 0) or (sending_cl not in selected_sending and len(selected_sending) > 0) or (receiving_cl not in selected_receiving and len(selected_receiving) > 0):
            continue
    
        # Sending to ligand
        source_node.append(sending_nodes_dict[sending_cl] + sending_offset)
        target_node.append(ligand_nodes_dict[ligand] + ligand_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # ligand to receptor
        source_node.append(ligand_nodes_dict[ligand] + ligand_offset)
        target_node.append(receptor_nodes_dict[receptor] + receptor_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # receptor to tf
        source_node.append(receptor_nodes_dict[receptor] + receptor_offset)
        target_node.append(tf_nodes_dict[tf] + tf_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # tf to receiving
        source_node.append(tf_nodes_dict[tf] + tf_offset)
        target_node.append(receiving_nodes_dict[receiving_cl] + receiving_offset)
        weight.append(1)
        line_label.append(l_c)
    
        l_c += 1
    
    # Feed into plotting
    calculated_opacity = determine_opacity(line_label)
    # calculated_linkcolor = "rgba(0, 0, 20," + str(calculated_opacity) + ")"
    # blue, blue

    calculated_linkcolor = "rgba(255,127,0," + str(calculated_opacity) + ")"
    # blue, blue

    domino_river_plot_helper('selected', node_label, source_node, target_node, weight, line_label, selection=[selected_ligands, selected_receptors, selected_tfs, selected_sending, selected_receiving], nodecolor='blue', linkcolor=calculated_linkcolor, linkhovercolor="blue", show_selection = False)


In [136]:
selected_sending = Immune_Cells
selected_receiving = ["Macrophages_Pos"]
selected_ligands=[]
selected_receptors=[]
selected_tfs=[]

domino_river_plot_selected(domino_df, selected_ligands, selected_receptors, selected_tfs, selected_sending, selected_receiving)

### River Plots - Multiple Selections, sans TFs

Finally, in an effort to simplify the river plots when analyzing a wide range of parameters, we included a version where the TFs are not plotted.

In [137]:
# Dropped TF version
def domino_river_plot_selected_notf(domino_df, selected_ligands, selected_receptors, selected_sending, selected_receiving):

    domino_output_list = get_domino_list(domino_df)

    # Generate list of all nodes in each set
    sending_nodes = sorted(list(set([l[6] for l in domino_output_list])))
    ligand_nodes = sorted(list(set([l[0] for l in domino_output_list])))
    receptor_nodes = sorted(list(set([l[1] for l in domino_output_list])))
    receiving_nodes = sorted(list(set([l[7] for l in domino_output_list])))
    
    # Dictionaries of positions
    sending_nodes_dict = dict(zip(sending_nodes, range(len(sending_nodes))))
    ligand_nodes_dict = dict(zip(ligand_nodes, range(len(ligand_nodes))))
    receptor_nodes_dict = dict(zip(receptor_nodes, range(len(receptor_nodes))))
    receiving_nodes_dict = dict(zip(receiving_nodes, range(len(receiving_nodes))))
    
    # Total list of nodes
    nodes = sending_nodes + ligand_nodes + receptor_nodes + receiving_nodes
    
    # Computed offsets to position
    sending_offset = 0
    ligand_offset = sending_offset + len(sending_nodes)
    receptor_offset = ligand_offset + len(ligand_nodes)
    receiving_offset = receptor_offset + len(receptor_nodes)
    
    # Build graph
    node_label = nodes
    source_node = []
    target_node = []
    weight = []
    # Track which line in the domino object each link comes from
    line_label = []
    # Track each line in the domino object, so we can do the hovering later
    l_c = 0
    
    for line in domino_output_list:
        sending_cl = line[6]
        ligand = line[0]
        receptor = line[1]
        receiving_cl = line[7]
        tf = line[2]
        
        # Populate only for this tf
        if (ligand not in selected_ligands and len(selected_ligands) > 0) or (receptor not in selected_receptors and len(selected_receptors) > 0) or (sending_cl not in selected_sending and len(selected_sending) > 0) or (receiving_cl not in selected_receiving and len(selected_receiving) > 0):
            continue
    
        # Sending to ligand
        source_node.append(sending_nodes_dict[sending_cl] + sending_offset)
        target_node.append(ligand_nodes_dict[ligand] + ligand_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # Ligand to receptor
        source_node.append(ligand_nodes_dict[ligand] + ligand_offset)
        target_node.append(receptor_nodes_dict[receptor] + receptor_offset)
        weight.append(1)
        line_label.append(l_c)
    
        # Receptor to receiving
        source_node.append(receptor_nodes_dict[receptor] + receptor_offset)
        target_node.append(receiving_nodes_dict[receiving_cl] + receiving_offset)
        weight.append(1)
        line_label.append(l_c)
    
        l_c += 1

    # Feed into plotting

    # Automatically calculate opacity
    calculated_opacity = determine_opacity(line_label)
    
    # calculated_linkcolor = "rgba(0, 0, 20," + str(calculated_opacity) + ")"
    # blue, blue

    calculated_linkcolor = "rgba(255,127,0," + str(calculated_opacity) + ")"
    # blue, blue

    domino_river_plot_helper('tf', node_label, source_node, target_node, weight, line_label, selection='N/A',
                            nodelinecolor="black", nodelinewidth=0.1, nodecolor="blue", linkcolor=calculated_linkcolor, linkhovercolor="blue", show_title=False)

In [138]:
selected_ligands = []
selected_receptors = []
selected_sending = Immune_Cells
selected_receiving = ["Macrophages_Pos"]

domino_river_plot_selected_notf(domino_df, selected_ligands, selected_receptors, selected_sending, selected_receiving)